# Route B (fly): polyploidy and redox in the Drosophila brain, from single-cell ATAC fragments

Choose **Runtime → Run all**. The step-by-step guide (PDF) explains every cell and screen.

In [ ]:
#@title Cell 1 - Setup, settings and connect Google Drive
import gzip, os, re, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.special import betaln, gammaln, logsumexp
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_b_fly"  #@param {type:"string"}
DATA_DIR = os.environ.get("ROUTE_B_DATA", DATA_DIR)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")


class StopHere(Exception):
    """Stops the notebook with a plain-language message."""


if not os.path.isdir(DATA_DIR):
    raise StopHere(f"Folder not found: {DATA_DIR}. Check the folder name in Google Drive and fix DATA_DIR above.")

#@markdown **Runs**: one sub-folder per sequencing run, each holding fragments.tsv.gz and singlecell.csv.
#@markdown Prefer runs that include the optic lobes and flies at least 1-3 weeks old (see the guide).
RUNS = "FDM__f2501a__10x_scATAC_mix_DGRP_adult_brain, FDM__1d60cb__10x_scATAC_mix_DGRP_adult_central_brain, FDM__9c338b__10x_scATAC_mix_DGRP_adult_central_brain"  #@param {type:"string"}
GTF_FILE = "Drosophila_melanogaster.gtf.gz"  #@param {type:"string"}
N_CLUSTERS = 20  #@param {type:"integer"}
MIN_FRAGMENTS = 2000  #@param {type:"integer"}
MAX_FRAGMENTS = 60000  #@param {type:"integer"}
GENES_OF_INTEREST = "geminin, dup, Cdc6, Orc1"  #@param {type:"string"}
N_RANDOM_SETS = 500  #@param {type:"integer"}
N_PERMUTATIONS = 2000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}

FDR_MAX, RANDOM_P_MAX, CONTROL_P_MAX = 0.05, 0.05, 0.05
PLOIDIES = (2, 4, 8, 16)
AUTOSOMES = {"2L", "2R", "3L", "3R"}
BIN = 500            # base pairs per bin for gene accessibility
PROMOTER = 1000      # base pairs upstream of each gene included
rng = np.random.default_rng(SEED)
RUN_LIST = [r.strip() for r in RUNS.split(",") if r.strip()]
GOI = [g.strip() for g in re.split(r"[,\s;]+", GENES_OF_INTEREST) if g.strip()]
print("Runs:", RUN_LIST, "| genes of interest:", GOI, "| data folder:", DATA_DIR)

In [ ]:
#@title Cell 2 - Check the data files in your Drive folder
paths, missing = {}, []
for run in RUN_LIST:
    for name in ("fragments.tsv.gz", "singlecell.csv"):
        p = os.path.join(DATA_DIR, run, name)
        if os.path.exists(p):
            print(f"   found   {run}/{name}  ({os.path.getsize(p) / 1e6:.0f} MB)")
            paths[(run, name)] = p
        else:
            print(f"   MISSING {run}/{name}")
            missing.append(f"{run}/{name}")
gtf = os.path.join(DATA_DIR, GTF_FILE)
if os.path.exists(gtf):
    print(f"   found   {GTF_FILE}  ({os.path.getsize(gtf) / 1e6:.0f} MB)")
else:
    print(f"   MISSING {GTF_FILE}")
    missing.append(GTF_FILE)
if missing:
    raise StopHere(f"{len(missing)} file(s) missing. Check folder and file names (see the guide), then run this cell again.")
print("All files ready.")

In [ ]:
#@title Cell 3 - Cells and genes
def norm_chrom(c):
    return re.sub(r"^chr", "", str(c))


cells = []
for run in RUN_LIST:
    sc = pd.read_csv(paths[(run, "singlecell.csv")])
    sc = sc[sc["barcode"] != "NO_BARCODE"]
    flag = "is__cell_barcode" if "is__cell_barcode" in sc.columns else None
    called = sc[sc[flag] == 1] if flag else sc
    cells.append(pd.DataFrame({"run": run, "barcode": called["barcode"].astype(str), "n_recovered": len(called)}))
cells = pd.concat(cells, ignore_index=True)
cells["cell_id"] = cells["run"] + ":" + cells["barcode"]
print(f"Cells called by Cell Ranger: {len(cells):,} in {len(RUN_LIST)} run(s)")

rows = []
with gzip.open(gtf, "rt") as handle:
    for line in handle:
        if line.startswith("#"):
            continue
        f = line.split("\t", 9)
        if len(f) < 9 or f[2] != "gene":
            continue
        name = re.search(r'gene_name "([^"]+)"', f[8]) or re.search(r'gene_id "([^"]+)"', f[8])
        rows.append((norm_chrom(f[0]), int(f[3]) - 1, int(f[4]), f[6], name.group(1)))
genes = pd.DataFrame(rows, columns=["chrom", "start", "end", "strand", "gene"]).drop_duplicates("gene")
genes = genes[genes.chrom.isin(AUTOSOMES | {"X", "4"})].reset_index(drop=True)
genes["w_start"] = np.where(genes.strand == "+", genes.start - PROMOTER, genes.start).clip(0)
genes["w_end"] = np.where(genes.strand == "-", genes.end + PROMOTER, genes.end)
print(f"Genes read from the annotation: {len(genes):,}")

In [ ]:
#@title Cell 4 - Read fragments: overlap counts and gene accessibility (10-30 minutes per run)
K = 6
SHIFT = np.uint64(32)


def count_overlaps(codes, starts, ends, n, k=K):
    """Per cell, f_x = number of fragments whose left end is covered by x same-cell fragments."""
    codes, starts, ends = (np.asarray(a, dtype=np.uint64) for a in (codes, starts, ends))
    if codes.size == 0:
        return np.zeros((n, k), dtype=np.int64)
    key = (codes << SHIFT) | starts
    order = np.lexsort((ends, key))
    key, ends, codes = key[order], ends[order], codes[order]
    keep = np.r_[True, (key[1:] != key[:-1]) | (ends[1:] != ends[:-1])]
    key, ends, codes = key[keep], ends[keep], codes[keep]
    end_key = np.sort((codes << SHIFT) | ends)
    depth = np.searchsorted(key, key, side="right") - np.searchsorted(end_key, key, side="right")
    depth = np.clip(depth, 1, k).astype(np.int64)
    return np.bincount(codes.astype(np.int64) * k + depth - 1, minlength=n * k).reshape(n, k)


chrom_bins = {}
offset = 0
for c in sorted(set(genes.chrom)):
    size = int(genes.loc[genes.chrom == c, "w_end"].max()) + 10 * BIN
    chrom_bins[c] = (offset, size // BIN + 1)
    offset += size // BIN + 1
n_bins = offset

overlaps, bin_blocks = [], []
for run in RUN_LIST:
    run_cells = cells[cells.run == run]
    index = pd.Index(run_cells["barcode"])
    f_counts = np.zeros((len(index), K), dtype=np.int64)
    acc = sp.csr_matrix((len(index), n_bins), dtype=np.float32)
    reader = pd.read_csv(paths[(run, "fragments.tsv.gz")], sep="\t", comment="#", header=None, usecols=[0, 1, 2, 3],
                         names=["chrom", "start", "end", "barcode"], dtype={"chrom": str, "barcode": str},
                         chunksize=5_000_000)
    buffer, current = [], None

    def flush(chrom, parts):
        global acc
        if not parts:
            return
        c = np.concatenate([p[0] for p in parts]); s = np.concatenate([p[1] for p in parts]); e = np.concatenate([p[2] for p in parts])
        if chrom in AUTOSOMES:
            f_counts[:] += count_overlaps(c, s, e, len(index))
        if chrom in chrom_bins:
            base, nb = chrom_bins[chrom]
            b = np.r_[np.minimum(s // BIN, nb - 1), np.minimum((e - 1) // BIN, nb - 1)].astype(np.int32) + base
            cc = np.r_[c, c].astype(np.int32)
            acc = acc + sp.csr_matrix((np.ones(b.size, dtype=np.float32), (cc, b)), shape=(len(index), n_bins))

    for chunk in reader:
        chunk["chrom"] = chunk["chrom"].map(norm_chrom)
        code = index.get_indexer(chunk["barcode"])
        chunk = chunk[code >= 0].assign(code=code[code >= 0])
        for chrom, part in chunk.groupby("chrom", sort=False):
            if chrom != current:
                flush(current, buffer)
                buffer, current = [], chrom
            buffer.append((part["code"].to_numpy(np.int32), part["start"].to_numpy(np.int64), part["end"].to_numpy(np.int64)))
    flush(current, buffer)
    overlaps.append(pd.DataFrame(f_counts, columns=[f"f{i}" for i in range(1, K + 1)], index=run_cells["cell_id"]))
    bin_blocks.append(acc)
    print(f"   {run}: {len(index):,} cells, {int(f_counts.sum()):,} autosomal fragments")

overlaps = pd.concat(overlaps).reindex(cells["cell_id"])
cut_bins = sp.vstack(bin_blocks).tocsr()

g_rows, g_cols = [], []
for gi, g in genes.iterrows():
    base, nb = chrom_bins[g.chrom]
    b0, b1 = base + g.w_start // BIN, base + min(g.w_end // BIN, nb - 1)
    g_rows.extend(range(b0, b1 + 1)); g_cols.extend([gi] * (b1 - b0 + 1))
membership = sp.csr_matrix((np.ones(len(g_rows), dtype=np.float32), (g_rows, g_cols)), shape=(n_bins, len(genes)))
gene_counts = (cut_bins @ membership).tocsr()
cells["n_fragments"] = overlaps.sum(axis=1).to_numpy()
cells["n_cuts"] = np.asarray(cut_bins.sum(axis=1)).ravel()
all_depth = cells[["run", "n_fragments"]].copy()   # every called cell, kept for the doublet check in Cell 7
print(f"Gene accessibility matrix: {gene_counts.shape[0]:,} cells x {gene_counts.shape[1]:,} genes")

In [ ]:
#@title Cell 5 - Group cells into clusters and name them from marker genes
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

keep = (cells.n_fragments >= MIN_FRAGMENTS) & (cells.n_fragments <= MAX_FRAGMENTS)
cells = cells[keep.to_numpy()].reset_index(drop=True)
overlaps = overlaps.loc[cells.cell_id]
gene_counts = gene_counts[np.flatnonzero(keep.to_numpy())]
print(f"Cells with {MIN_FRAGMENTS:,}-{MAX_FRAGMENTS:,} autosomal fragments: {len(cells):,}")

detected = np.asarray((gene_counts > 0).mean(axis=0)).ravel()
use = np.flatnonzero(detected > 0.01)
tf = sp.diags(1 / np.maximum(np.asarray(gene_counts[:, use].sum(axis=1)).ravel(), 1)) @ gene_counts[:, use]
tfidf = tf.multiply(np.log1p(len(cells) / np.maximum(detected[use] * len(cells), 1))).tocsr()
tfidf.data = np.log1p(tfidf.data * 1e4)
svd = TruncatedSVD(n_components=30, random_state=SEED).fit_transform(tfidf)[:, 1:]
svd = svd / np.linalg.norm(svd, axis=1, keepdims=True)
cells["cluster"] = KMeans(n_clusters=N_CLUSTERS, n_init=5, random_state=SEED).fit_predict(svd).astype(str)

MARKERS = {"neuron": ["elav", "nSyb", "brp", "fne"], "glia": ["repo"], "surface glia": ["moody", "Mdr65"],
           "astrocyte": ["alrm", "Eaat1"], "ensheathing glia": ["Eaat2"], "cortex glia": ["wrapper"]}
gene_pos = {g: i for i, g in enumerate(genes.gene)}

# Polyploid cells have more fragments. Thin every cell to the same number of cut sites in genes
# so that accessibility scores do not rise with depth alone.
in_genes = np.asarray(gene_counts.sum(axis=1)).ravel()
TARGET = int(np.quantile(in_genes, 0.1))
thin = gene_counts.tocsr(copy=True).astype(np.int64)
prob = np.repeat(TARGET / np.maximum(in_genes, 1), np.diff(thin.indptr))
thin.data = rng.binomial(thin.data, np.minimum(prob, 1.0)); thin.eliminate_zeros()
enough = in_genes >= TARGET
print(f"Thinned every cell to {TARGET:,} cut sites in genes; {int((~enough).sum()):,} cells below that are left out")
cells = cells[enough].reset_index(drop=True)
overlaps = overlaps.loc[cells.cell_id]
gene_counts, thin, svd = gene_counts[enough], thin[enough], svd[enough]
logcpm = thin.astype(float); logcpm.data = np.log1p(logcpm.data)
marker_table = {}
for label, ms in MARKERS.items():
    idx = [gene_pos[m] for m in ms if m in gene_pos]
    if idx:
        v = np.asarray(logcpm[:, idx].mean(axis=1)).ravel()
        marker_table[label] = pd.Series(v).groupby(cells.cluster).mean()
marker_table = pd.DataFrame(marker_table)
z = (marker_table - marker_table.mean()) / marker_table.std()
names = {}
for cl, row in z.iterrows():
    if row.get("surface glia", -9) > 1.5:
        names[cl] = "surface glia"
    elif row.get("glia", -9) > 0.5:
        sub = max(("astrocyte", "ensheathing glia", "cortex glia"), key=lambda k: row.get(k, -9))
        names[cl] = "glia" + (f": {sub.replace(' glia', '')}" if row.get(sub, -9) > 1 else "")
    else:
        names[cl] = "neuron"
cells["cell_type"] = cells.cluster.map(names) + " (c" + cells.cluster + ")"
summary = z.round(2).assign(name=pd.Series(names), cells=cells.cluster.value_counts())
display(summary.sort_values("name"))

In [ ]:
#@title Cell 6 - Ploidy of every cell, from overlapping fragments
GRID = np.exp(np.linspace(np.log(5e-4), np.log(0.7), 90)); LOG_GRID = np.log(GRID)


def _log_categories(p, kappa, artefact, k):
    t = p - 1; a, b = GRID * kappa, (1 - GRID) * kappa
    out = np.zeros((len(GRID), k))
    for s in range(t + 1):
        out[:, min(s, k - 1)] += np.exp(gammaln(t + 1) - gammaln(s + 1) - gammaln(t - s + 1)
                                        + betaln(s + a, t - s + b) - betaln(a, b))
    out = np.maximum(out + artefact[None, :], 1e-300)
    return np.log(out / out.sum(axis=1, keepdims=True))


def _basis(x, knots):
    return np.column_stack([np.ones_like(x), x] + [np.maximum(x - kn, 0) for kn in knots])


def _e_step(C, n, weights, th, kappa):
    lm = np.zeros((len(C), len(PLOIDIES))); ms = np.zeros_like(lm); vs = np.zeros_like(lm)
    for j, p in enumerate(PLOIDIES):
        lp = _log_categories(p, kappa, th["art"], C.shape[1])
        mean = _basis(np.log(2 * n / p), th["knots"]) @ th["coef"]
        joint = C @ lp.T - 0.5 * ((LOG_GRID[None, :] - mean[:, None]) / th["tau"]) ** 2
        norm = logsumexp(joint, axis=1, keepdims=True); post = np.exp(joint - norm)
        lm[:, j] = norm[:, 0]; ms[:, j] = post @ LOG_GRID; vs[:, j] = post @ LOG_GRID**2 - ms[:, j] ** 2
    lj = lm + np.log(weights)
    return np.exp(lj - logsumexp(lj, axis=1, keepdims=True)), ms, vs


def fit_ploidy(C, kappa=8.0, n_iter=40):
    C = np.asarray(C, float); n = C.sum(axis=1); k = C.shape[1]
    rate = np.maximum((C[:, 1] + 0.5) / (C[:, 0] + 1), 1e-4)
    knots = np.quantile(np.log(n), [0.2, 0.4, 0.6, 0.8])
    th = {"coef": np.linalg.lstsq(_basis(np.log(n), knots), np.log(rate), rcond=None)[0], "knots": knots,
          "tau": 0.5, "art": np.r_[0, 0, np.full(k - 2, 1e-4)]}
    w = np.r_[0.9, np.full(len(PLOIDIES) - 1, 0.1 / (len(PLOIDIES) - 1))]
    for _ in range(n_iter):
        resp, ms, vs = _e_step(C, n, w, th, kappa)
        new_w = np.maximum(resp.mean(0), 1e-5); new_w /= new_w.sum()
        X = np.concatenate([np.log(2 * n / p) for p in PLOIDIES]); Y = ms.T.ravel()
        W = resp.T.ravel(); V = vs.T.ravel(); B = _basis(X, knots); r = np.sqrt(W)
        coef = np.linalg.lstsq(B * r[:, None], Y * r, rcond=None)[0]
        tau = max(float(np.sqrt(np.average((Y - B @ coef) ** 2 + V, weights=W))), 0.05)
        d = resp[:, 0]; art = np.r_[0, 0, np.maximum((d[:, None] * C[:, 2:]).sum(0) / max((d * n).sum(), 1), 1e-7)]
        done = np.max(np.abs(new_w - w)) < 1e-4
        w, th = new_w, {"coef": coef, "knots": knots, "tau": tau, "art": art}
        if done:
            break
    resp, ms, _ = _e_step(C, n, w, th, kappa)
    return resp, (resp * ms).sum(1)


for p in PLOIDIES:
    cells[f"P_{p}C"] = np.nan
for cl, part in cells.groupby("cluster"):
    resp, log_s = fit_ploidy(overlaps.loc[part.cell_id].to_numpy())
    for j, p in enumerate(PLOIDIES):
        cells.loc[part.index, f"P_{p}C"] = resp[:, j]
    cells.loc[part.index, "per_copy_log_s"] = log_s
cells["P_polyploid"] = 1 - cells["P_2C"]
cells["expected_copies"] = sum(cells[f"P_{p}C"] * p for p in PLOIDIES)
cells["fragments_per_copy"] = cells.n_fragments * 2 / cells.expected_copies
best = cells[[f"P_{p}C" for p in PLOIDIES]].to_numpy()
cells["ploidy_call"] = np.where(best.max(1) >= 0.8, np.array([f"{p}C" for p in PLOIDIES])[best.argmax(1)], "uncertain")
display(pd.crosstab(cells.cell_type, cells.ploidy_call))

In [ ]:
#@title Cell 7 - Main check: more polyploidy than doublets can explain?
# Two cells captured together look like one cell with twice the DNA. About 0.8% of captured cells per 1,000
# recovered are doublets. Doublets have about twice the fragments, so the fragment filter in Cell 5 keeps them more
# often than single cells; and a doublet of two different neurons lands in a neuron cluster. So the expected
# doublet share is worked out for all kept cells of a run, not only for same-cluster pairs.
expected = {}
for run, part in all_depth.groupby("run"):
    d = part.n_fragments.to_numpy()
    D = min(len(d) / 1000 * 0.008, 0.5)
    pair = rng.choice(d, 20000) + rng.choice(d, 20000)
    pass_single = np.mean((d >= MIN_FRAGMENTS) & (d <= MAX_FRAGMENTS))
    pass_double = np.mean((pair >= MIN_FRAGMENTS) & (pair <= MAX_FRAGMENTS))
    kept_share = D * pass_double / (D * pass_double + (1 - D) * pass_single)
    expected[run] = kept_share
    print(f"{run[:45]}: {len(d):,} cells, median {np.median(d):,.0f} fragments; doublets expected "
          f"{100 * D:.1f}% of all cells, {100 * kept_share:.1f}% of the cells kept by the fragment filter")
cells["expected_doublet"] = cells.run.map(expected)
rows = []
for ct, part in cells.groupby("cell_type"):
    boot = [part.P_polyploid.sample(len(part), replace=True, random_state=i).mean() for i in range(500)]
    rows.append({"cell_type": ct, "cells": len(part), "polyploid_rate": part.P_polyploid.mean(),
                 "ci_low": np.quantile(boot, 0.025), "expected_doublets": part.expected_doublet.mean()})
check = pd.DataFrame(rows).set_index("cell_type")
check["excess"] = check.polyploid_rate - check.expected_doublets
check["clear_excess"] = check.ci_low > check.expected_doublets
display(check.sort_values("polyploid_rate", ascending=False).round(3))
overall = cells.P_polyploid.mean(); expected = cells.expected_doublet.mean()
print(f"All cells: polyploid rate {overall:.3f}, expected from doublets {expected:.3f}")
print(f"Cell types with clearly more polyploidy than doublets: {int(check.clear_excess.sum())} of {len(check)}")
if check.clear_excess.sum() == 0:
    print("WARNING: no cell type has more polyploidy than doublets would give. Later results cannot be trusted; "
          "use deeper runs, older flies, or doublet-removed data.")
surface = cells.cell_type.str.startswith("surface glia")
if surface.any():
    print(f"Bonus check: surface glia polyploid rate {cells.P_polyploid[surface].mean():.2f} "
          f"(expected high), other cells {cells.P_polyploid[~surface].mean():.2f}")
else:
    print("No surface glia cluster (expected: they are very few). The doublet check above is the main test.")

In [ ]:
#@title Cell 8 - Gene programmes (fly gene symbols)
PROGRAMMES = {
    "REDOX: cnc (NRF2) pathway": "cnc Keap1 Gclc Gclm GstD1 GstD2 GstE1 GstS1 Zw Pgd Men",
    "REDOX: thioredoxin/peroxiredoxin": "Trxr-1 Trxr-2 Trx-2 Jafrac1 Jafrac2 Prx3 Prx5 PHGPx",
    "REDOX: antioxidant enzymes": "Sod1 Sod2 Cat Eip71CD SelR Ogg1",
    "REDOX: iron": "Fer1HCH Fer2LCH Irp1A Tsf1 Mvl",
    "REDOX: mitochondrial respiration": "ND-75 ND-49 ND-30 ND-42 SdhA SdhB UQCR-C1 UQCR-C2 Cyt-c-p COX4 COX5A ATPsynbeta blw",
    "EXPLORE: licensing and cell cycle re-entry": "geminin dup Cdc6 Orc1 Mcm2 Mcm3 dpa Mcm5 Mcm6 Mcm7 CycE E2f1 Myc CycD",
    "EXPLORE: p53 head DNA damage response": "Ku80 Irbp Cht8 CG3344 CG4734 Fanci lok rad50 Xrp1 p53",
    "CONTROL: unfolded protein response": "Hsc70-3 Xbp1 crc PEK Ire1 Atf6 Pdi ERp60 Calr",
    "CONTROL: heat shock": "Hsp70Aa Hsp83 Hsp68 Hsp27 Hsp26 Hsc70-4 DnaJ-1 Hsf",
    "CONTROL: DNA damage": "p53 mei-41 tefu lok spn-A mus304 Xrp1 rpr hid",
    "CONTROL: immediate-early genes": "Hr38 sr kay Jra puc CrebA",
}
# Some genes are listed under a different name in the annotation, so each one is looked up under its synonyms too.
# FlyBase calls Geminin "geminin" (CG3183); "Gem" is only a synonym, which is why earlier runs could not find it.
ALIASES = {"geminin": ["Gem", "CG3183"], "dup": ["Cdt1"], "dpa": ["Mcm4"], "Zw": ["G6PD", "G6pd"],
           "Ero1L": ["Ero1", "CG1333"], "PHGPx": ["CG12013"], "Gss1": ["Gss"], "Gr1": ["Gr", "GR"],
           "Trxr-1": ["Trxr1"], "Jafrac1": ["Prx2"], "Men-b": ["Men-b", "CG5889"]}


def resolve(g, present):
    if g in present:
        return g
    for a in ALIASES.get(g, []):
        if a in present:
            return a
    return None


_renamed, _resolved = {}, {}
for n, v in PROGRAMMES.items():
    out = []
    for g in v.split():
        r = resolve(g, gene_pos)
        if r:
            out.append(r)
            if r != g:
                _renamed[g] = r
    _resolved[n] = out
PROGRAMMES = _resolved
if _renamed:
    print("Found under a synonym:", ", ".join(f"{k} -> {v}" for k, v in _renamed.items()))
for n, v in PROGRAMMES.items():
    print(f"{n}: {len(v)} genes found")
PROGRAMMES = {n: v for n, v in PROGRAMMES.items() if len(v) >= 4}
missing_goi = [g for g in GOI if g not in gene_pos]
if missing_goi:
    print("Genes of interest not found in the annotation (check FlyBase symbols):", missing_goi)
GOI = [g for g in GOI if g in gene_pos]

In [ ]:
#@title Cell 9 - Scores and tools
lognorm = logcpm.tocsc()


def gene_z(idx):
    """Per-gene accessibility (log CPM), z-scored within cluster."""
    m = np.asarray(lognorm[:, idx].todense(), float)
    out = np.empty_like(m)
    for cl in cells.cluster.unique():
        r = (cells.cluster == cl).to_numpy()
        sd = m[r].std(0); out[r] = (m[r] - m[r].mean(0)) / np.where(sd > 0, sd, 1)
    return out


def programme_score(gs):
    return gene_z([gene_pos[g] for g in gs]).mean(1)


scores = pd.DataFrame({n: programme_score(v) for n, v in PROGRAMMES.items()})
lfpc = np.log(cells.fragments_per_copy.to_numpy()); lfpc = (lfpc - lfpc.mean()) / lfpc.std()
dummies = pd.get_dummies(cells.cluster + "|" + cells.run).astype(float).to_numpy()
basis, _ = np.linalg.qr(np.column_stack([dummies, lfpc, lfpc**2]))
strata = (cells.cluster + "|" + cells.run).to_numpy()


def residual(v):
    v = np.asarray(v, float); return v - basis @ (basis.T @ v)


def corr(a, b):
    a, b = a - a.mean(0), b - b.mean(0)
    return (a * b).sum(0) / np.sqrt((a**2).sum(0) * (b**2).sum(0))


def perm_p(obs, null):
    return (1 + np.sum(np.abs(null) >= abs(obs))) / (len(null) + 1)


def fdr(p):
    p = np.asarray(p, float); o = np.argsort(p); a = p[o] * len(p) / np.arange(1, len(p) + 1)
    out = np.empty_like(p); out[o] = np.minimum(np.minimum.accumulate(a[::-1])[::-1], 1); return out


x = residual(cells.P_polyploid.to_numpy())
xs = np.tile(cells.P_polyploid.to_numpy()[:, None], (1, N_PERMUTATIONS))
for s in np.unique(strata):
    idx = np.flatnonzero(strata == s)
    for j in range(N_PERMUTATIONS):
        xs[idx, j] = xs[rng.permutation(idx), j]
xs = residual(xs)
print("Tools ready.")

In [ ]:
#@title Cell 10 - Screens 1-3: programmes vs ploidy, random genes, stress controls
rows = []
for n in scores.columns:
    r = residual(scores[n].to_numpy())
    rho = float(corr(x[:, None], r[:, None])[0])
    rows.append({"programme": n, "genes": len(PROGRAMMES[n]), "partial_rho": rho,
                 "p_perm": perm_p(rho, corr(xs, np.repeat(r[:, None], xs.shape[1], 1)))})
result = pd.DataFrame(rows)
for family in ("REDOX", "EXPLORE"):
    m = result.programme.str.startswith(family)
    if m.any():
        result.loc[m, "q_fdr"] = fdr(result.loc[m, "p_perm"])

level = np.asarray(lognorm.mean(axis=0)).ravel()
ok = np.flatnonzero(level > 0)
bins = pd.qcut(pd.Series(level[ok]).rank(method="first"), 20, labels=False).to_numpy()
bin_of = dict(zip(ok, bins))
avoid = {gene_pos[g] for v in PROGRAMMES.values() for g in v}
by_bin = {b: [g for g, bb in zip(ok, bins) if bb == b and g not in avoid] for b in range(20)}
for n in [n for n in scores.columns if n.startswith(("REDOX", "EXPLORE"))]:
    idx = [gene_pos[g] for g in PROGRAMMES[n]]
    null = [corr(x[:, None], residual(gene_z([rng.choice(by_bin[bin_of.get(i, 0)]) for i in idx]).mean(1))[:, None])[0]
            for _ in range(N_RANDOM_SETS)]
    i = result.programme == n
    result.loc[i, "random_low"], result.loc[i, "random_high"] = np.quantile(null, [0.025, 0.975])
    result.loc[i, "p_vs_random"] = perm_p(result.loc[i, "partial_rho"].iloc[0], np.array(null))

controls = [n for n in scores.columns if n.startswith("CONTROL")]
for n in [n for n in scores.columns if n.startswith(("REDOX", "EXPLORE"))]:
    i = result.programme == n; rho = result.loc[i, "partial_rho"].iloc[0]; worst, which = 0.0, ""
    for c in controls:
        r = residual(scores[n].to_numpy() - scores[c].to_numpy())
        obs = float(corr(x[:, None], r[:, None])[0])
        p = 1.0 if np.sign(obs) != np.sign(rho) else perm_p(obs, corr(xs, np.repeat(r[:, None], xs.shape[1], 1)))
        if p >= worst:
            worst, which = p, c.replace("CONTROL: ", "")
    result.loc[i, "p_vs_controls"], result.loc[i, "closest_control"] = worst, which
display(result.round(3))
passes = (result.q_fdr < FDR_MAX) & (result.p_vs_random < RANDOM_P_MAX) & (result.p_vs_controls < CONTROL_P_MAX)
result["verdict"] = np.where(result.programme.str.startswith("CONTROL"), "(control)",
                    np.where(passes, np.where(result.partial_rho > 0, "passes: more open in polyploid cells",
                                              "passes: less open in polyploid cells"),
                    np.where(result.q_fdr < FDR_MAX, "not specific", "no link")))
print()
for _, r in result[~result.programme.str.startswith("CONTROL")].iterrows():
    print(f"{r.programme}: {r.verdict} (partial rho {r.partial_rho:+.3f})")

In [ ]:
#@title Cell 11 - Screen 4: gene by gene, including Geminin and dup
tested = list(dict.fromkeys([g for v in PROGRAMMES.values() for g in v] + GOI))
Z = np.column_stack([residual(col) for col in gene_z([gene_pos[g] for g in tested]).T])
Zs = (Z - Z.mean(0)) / np.where(Z.std(0) > 0, Z.std(0), 1)
xz = (x - x.mean()) / x.std(); xsz = (xs - xs.mean(0)) / xs.std(0)
rho = xz @ Zs / len(xz); null = xsz.T @ Zs / len(xz)
genes_result = pd.DataFrame({"gene": tested, "partial_rho": rho,
                             "p_perm": (1 + (np.abs(null) >= np.abs(rho)).sum(0)) / (null.shape[0] + 1)})
genes_result["q_fdr"] = fdr(genes_result.p_perm)
genes_result["group"] = [", ".join(n.split(": ")[1] for n, v in PROGRAMMES.items() if g in v) or "gene of interest"
                         for g in tested]
genes_result = genes_result.assign(_a=genes_result.partial_rho.abs()).sort_values(["p_perm", "_a"], ascending=[True, False]).drop(columns="_a")
display(genes_result.head(15).round(3))
display(genes_result[genes_result.gene.isin(GOI)].round(3))

In [ ]:
#@title Cell 12 - Per cell type, charts and saving
per_type = []
for ct, part in cells.groupby("cell_type"):
    m = (cells.cell_type == ct).to_numpy()
    if m.sum() < 100 or cells.P_polyploid[m].std() < 0.02:
        continue
    for n in scores.columns:
        per_type.append({"cell_type": ct, "programme": n, "cells": int(m.sum()),
                         "partial_rho": float(corr(x[m, None], residual(scores[n].to_numpy())[m, None])[0])})
per_type = pd.DataFrame(per_type)
if not per_type.empty:
    display(per_type.pivot(index="programme", columns="cell_type", values="partial_rho").round(2))

fig, ax = plt.subplots(figsize=(7, 4.5))
order = cells.groupby("cell_type").P_polyploid.mean().sort_values()
ax.barh(order.index, order.values, color=["#c26a2e" if s.startswith("surface") else "#3b6ea8" for s in order.index])
ax.set_xlabel("Mean probability of being polyploid"); ax.set_title("Polyploidy by cell type", loc="left")
plt.tight_layout(); plt.savefig(os.path.join(DATA_DIR, "chart_ploidy_by_type.png"), dpi=150); plt.show()

for name, frame in {"route_b_cells.csv": cells, "route_b_programmes.csv": result, "route_b_genes.csv": genes_result,
                    "route_b_per_type.csv": per_type}.items():
    frame.to_csv(os.path.join(DATA_DIR, name), index=False)
print("Saved results to", DATA_DIR)